In [26]:
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

In [61]:
loader=PyPDFLoader(
    "../data/medical_report.pdf",
)
docs=loader.load()



In [62]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(docs)

In [63]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")


In [64]:
vectors=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [65]:
query = "Machine Learnings and Data Science Content"
data=vectors.similarity_search(query=query)
len(data)

4

In [66]:
context=""
for doc in data:
    context+=doc.page_content+"\n"
llm=ChatGroq(model="openai/gpt-oss-20b")


## chain -context-generate|prompt|llm|res

In [67]:
def get_context(query:str):
    data=vectors.similarity_search(query=query)
    context=""
    for doc in data:
        context+=doc.page_content+"\n"
    return {
        "context":context,
        "question":query
    }

In [68]:
from langchain_core.prompts import PromptTemplate
prompt=PromptTemplate.from_template("""
    You are a helpful assistant and provide anseer based on the context of the user if u dont know the answer then you can say i dont know
    context:{context}
    question:{question}
    """)

In [69]:
rag_chain=get_context | prompt | llm

In [78]:
res=rag_chain.invoke("what is the value of rbc count and is it in range")

In [79]:
print(res.content)

The RBC (red blood cell) count is **4.47 million cells per cubic millimeter (mm³)**.  
According to the reference interval given (3.80 – 4.80 million/mm³), this value is **within the normal range**.
